# PageIndex Flash Tree and Cache Check

This checkpoint builds a hierarchical tree for the real sample PDF with PageIndex Flash. The default build uses `summary=False` and `optimize=False`, so it does not call Groq or another LLM provider.

In [65]:
from pathlib import Path
import json
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the ConversIQ project or notebooks directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CACHE_DIR, SAMPLE_DOCS_DIR
from src.extraction import generate_doc_id
from src.indexer import build_tree, load_tree

pdf_path = SAMPLE_DOCS_DIR / 'harry_potter.pdf'
if not pdf_path.is_file():
    raise FileNotFoundError(f'Expected the manual test PDF at: {pdf_path}')

doc_id = generate_doc_id(pdf_path)
cache_path = CACHE_DIR / f'{doc_id}.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'PDF path: {pdf_path}')
print(f'Document ID: {doc_id}')
print(f'Cache path: {cache_path}')

Project root: D:\conversIQ_v3
PDF path: D:\conversIQ_v3\data\sample_docs\harry_potter.pdf
Document ID: 880c366958602a7e
Cache path: D:\conversIQ_v3\cache\trees\880c366958602a7e.json


## Test 1 — Build or reuse standardized cache

A missing standardized cache is built once. If it already exists, this test reuses it rather than rebuilding the document.

In [66]:
cache_existed_before = cache_path.is_file()
if cache_existed_before:
    print("Existing standardized cache found; reusing it.")
else:
    print("No standardized cache found; building it now.")

build_started = time.perf_counter()
tree = build_tree(pdf_path, doc_id)
build_elapsed = time.perf_counter() - build_started

assert isinstance(tree, dict)
assert cache_path.is_file()
print(f'Build/load elapsed: {build_elapsed:.2f} seconds')
print(f'Cache now exists: {cache_path.is_file()}')
print(f'Cache was newly created: {not cache_existed_before}')

Existing standardized cache found; reusing it.
Build/load elapsed: 0.01 seconds
Cache now exists: True
Cache was newly created: False


## Test 2 — Inspect the tree

The view below intentionally shows only a small representative portion of the native PageIndex hierarchy.

In [67]:
def compact_node(node: dict, depth: int = 1) -> dict:
    visible = {key: node[key] for key in ('title', 'node_id', 'start_index', 'end_index', 'key_items') if key in node}
    children = node.get('nodes', [])
    if depth > 0 and isinstance(children, list):
        visible['nodes'] = [compact_node(child, depth - 1) for child in children[:3] if isinstance(child, dict)]
    return visible

structure = tree.get('structure', [])
print(f'Top-level type: {type(tree).__name__}')
print(f'Top-level keys: {sorted(tree.keys())}')
print(f'Document title: {tree.get("doc_title") or tree.get("doc_name")}')
print(f'Major sections/nodes: {len(structure) if isinstance(structure, list) else "not available"}')
if isinstance(structure, list) and structure:
    print('Representative hierarchy:')
    print(json.dumps(compact_node(structure[0]), ensure_ascii=False, indent=2))
else:
    print('No top-level structure nodes were returned.')

Top-level type: dict
Top-level keys: ['doc_name', 'doc_title', 'has_abstract_or_references_section', 'structure', 'toc_source']
Document title: year at Hogwarts Academy where he is mysteriously entered in an unusual contest that challenges his wizarding skills, friendships and character, amid signs that an old enemy is growing stronger.
Major sections/nodes: 9
Representative hierarchy:
{
  "title": "LOGO",
  "node_id": "0000",
  "start_index": 5,
  "end_index": 7,
  "nodes": []
}


### Manual hierarchy inspection

### Manual hierarchy inspection

This compact view lists every top-level node and descendants through depth 3. It uses the existing `tree` value only; it does not call PageIndex or Groq.

In [60]:
def child_nodes(node: dict) -> list[dict]:
    children = node.get('nodes')
    return [child for child in children if isinstance(child, dict)] if isinstance(children, list) else []

def node_label(node: dict) -> str:
    title = node.get('title') or '<untitled>'
    node_id = node.get('node_id', '<missing>')
    start = node.get('start_index', '<missing>')
    end = node.get('end_index', '<missing>')
    return f'{title} (id={node_id}, pages={start}-{end}, children={len(child_nodes(node))})'

def print_hierarchy(nodes: list[dict], depth: int = 0, max_depth: int = 3) -> None:
    for node in nodes:
        print(f'{"  " * (depth - 1)}- {node_label(node)}')
        if depth < max_depth:
            print_hierarchy(child_nodes(node), depth + 1, max_depth)

def hierarchy_metrics(nodes: list[dict], depth: int = 1) -> tuple[int, int]:
    total = 0
    maximum_depth = 0
    for node in nodes:
        total += 1
        maximum_depth = max(maximum_depth, depth)
        child_total, child_depth = hierarchy_metrics(child_nodes(node), depth + 1)
        total += child_total
        maximum_depth = max(maximum_depth, child_depth)
    return total, maximum_depth

top_level_nodes = [node for node in structure if isinstance(node, dict)] if isinstance(structure, list) else []
total_nodes, maximum_depth = hierarchy_metrics(top_level_nodes)
print('Hierarchy (display depth: 3):')
print_hierarchy(top_level_nodes, depth=1)
print(f'\nTotal nodes: {total_nodes}')
print(f'Top-level nodes: {len(top_level_nodes)}')
print(f'Maximum hierarchy depth: {maximum_depth}')

Hierarchy (display depth: 3):
- LOGO (id=0000, pages=5-7, children=0)
- The Riddle House (id=0001, pages=7-22, children=1)
  - Chapter 1  (id=0002, pages=7-22, children=1)
    - The Riddle House (id=0003, pages=7-22, children=0)
- The Scar (id=0004, pages=22-32, children=1)
  - Chapter 2  (id=0005, pages=22-32, children=1)
    - The Scar (id=0006, pages=22-32, children=0)
- The Invitation (id=0007, pages=32-44, children=1)
  - Chapter 3  (id=0008, pages=32-44, children=1)
    - The Invitation (id=0009, pages=32-44, children=1)
- Back to the Burrow (id=0011, pages=44-56, children=1)
  - Chapter 4  (id=0012, pages=44-56, children=1)
    - Back to the Burrow (id=0013, pages=44-56, children=0)
- Weasleys’ Wizard Wheezes (id=0014, pages=56-70, children=1)
  - Chapter 5  (id=0015, pages=56-70, children=1)
    - Weasleys’ Wizard Wheezes (id=0016, pages=56-70, children=0)
- The Portkey (id=0017, pages=70-80, children=1)
  - Chapter 6  (id=0018, pages=70-80, children=1)
    - The Portkey (id=00

## Test 3 — Cache hit

In [61]:
cached_started = time.perf_counter()
cached_tree = build_tree(pdf_path, doc_id, force_rebuild=False)
cached_elapsed = time.perf_counter() - cached_started

assert cached_tree == tree
print('Disk cache was used (force_rebuild=False).')
print(f'Cached call elapsed: {cached_elapsed:.4f} seconds')
print(f'Cache file exists: {cache_path.is_file()}')
print(f'Returned top-level keys: {sorted(cached_tree.keys())}')

Disk cache was used (force_rebuild=False).
Cached call elapsed: 0.0015 seconds
Cache file exists: True
Returned top-level keys: ['doc_name', 'doc_title', 'has_abstract_or_references_section', 'structure', 'toc_source']


## Test 4 — Cache-only load

In [62]:
loaded_tree = load_tree(doc_id)
assert loaded_tree == tree
print('load_tree(doc_id) returned the cached native PageIndex structure successfully.')

load_tree(doc_id) returned the cached native PageIndex structure successfully.


## Test 5 — LLM-free default

`build_tree()` uses `page_index_flash(pdf_path, summary=False, optimize=False)`. This default path creates no Groq client and makes no Groq/API request.

In [63]:
print('Default PageIndex Flash build: LLM-free (summary=False, optimize=False).')
print('No Groq client or API call is used by this notebook build path.')

Default PageIndex Flash build: LLM-free (summary=False, optimize=False).
No Groq client or API call is used by this notebook build path.


## Optional — force rebuild

Set `RUN_FORCE_REBUILD` to `True` and run this cell manually only when a rebuild is specifically needed. The current implementation remains LLM-free, but this is deliberately not part of the normal checkpoint.

In [64]:
RUN_FORCE_REBUILD = False

if RUN_FORCE_REBUILD:
    rebuilt_tree = build_tree(pdf_path, doc_id, force_rebuild=True)
    print(f'Forced rebuild completed; top-level keys: {sorted(rebuilt_tree.keys())}')
else:
    print('Forced rebuild skipped. Set RUN_FORCE_REBUILD = True to run it manually.')

Forced rebuild skipped. Set RUN_FORCE_REBUILD = True to run it manually.
